> ## 🎙️ SPEAKER NOTES — Opening
> - Start with your screen showing the notebook, server terminal open in a split pane if possible.
> - Say: *"This is a backend service that takes a topic as input and produces a complete, publish-ready SEO article — fully validated. Let me walk you through the structure, the pipeline, and then a live run."*
> - Keep the intro to under 30 seconds before moving to the folder structure.

# SEO Article Generator — Demo

A backend service that takes a topic and returns a fully structured, SEO-validated article by analysing the top 10 Google search results and generating content through a multi-stage LLM pipeline.

**Stack:** FastAPI · LangGraph · Claude Sonnet · SerpAPI · SQLite

---
## 1. Project Structure

In [ ]:
import os

DESCRIPTIONS = {
    "main.py":             "FastAPI app — endpoints, lifespan, background job runner",
    "graph":               "LangGraph pipeline (builder + 6 node files)",
    "builder.py":          "Wires nodes into a compiled StateGraph",
    "state.py":            "ArticleState TypedDict shared across all nodes",
    "nodes":               "One file per pipeline stage",
    "models":              "Pydantic schemas for every layer (inputs → SERP → outline → article)",
    "prompts":             "System + user prompt builders, kept separate from node logic",
    "services":            "SerpAPI wrapper + deterministic mock for testing",
    "db":                  "SQLite job tracking — status, stage, result JSON",
    "tests":               "10 SEO constraint tests run against a fixture",
}

SKIP = {"__pycache__", ".venv", "aiseo_env", ".git",
        "checkpoints.db", "jobs.db", ".env", "requirements.txt",
        "__init__.py", ".gitignore", "fixtures"}

def _tree(root, prefix="", depth=0, max_depth=2):
    if depth > max_depth:
        return
    try:
        items = sorted(
            [i for i in os.listdir(root) if i not in SKIP and not i.endswith(".pyc")],
            key=lambda x: (os.path.isfile(os.path.join(root, x)), x)
        )
    except PermissionError:
        return
    for idx, name in enumerate(items):
        path = os.path.join(root, name)
        connector = "└── " if idx == len(items) - 1 else "├── "
        desc = DESCRIPTIONS.get(name, "")
        suffix = f"  ← {desc}" if desc else ""
        print(f"{prefix}{connector}{name}{suffix}")
        if os.path.isdir(path):
            extension = "    " if idx == len(items) - 1 else "│   "
            _tree(path, prefix + extension, depth + 1, max_depth)

seo_agent_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "seo_agent")
print("seo_agent/")
_tree(seo_agent_path)

> ## 🎙️ SPEAKER NOTES — Structure
> - Point out that each concern is isolated: nodes are thin — they call into prompts, models, or services, never each other.
> - Mention the two databases: `jobs.db` is app-owned (job status + result JSON), `checkpoints.db` is LangGraph-owned (full state per node). Never share schema between them.
> - Say: *"The `graph/` folder is the heart of it — one file per pipeline stage, which makes each step independently testable."*

---
## 2. Pipeline Architecture

The article is generated through a 6-node LangGraph pipeline. Each node reads from a shared `ArticleState` TypedDict and writes back a partial update — LangGraph merges updates immutably between steps and checkpoints state to SQLite after every node.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "seo_agent"))

from langgraph.checkpoint.memory import MemorySaver
from graph.builder import build_graph
from IPython.display import Image

graph = build_graph(MemorySaver())
Image(graph.get_graph().draw_mermaid_png())

### Node Responsibilities

| Node | Role | LLM? |
|---|---|---|
| **serp_fetch** | Fetches the top-10 Google results for the topic via SerpAPI (or mock). Stores titles, URLs, snippets, and People Also Ask questions. | No |
| **analyze_serp** | Reads the SERP results and extracts the primary keyword, 5–8 secondary keywords, common subtopics, content format (guide/listicle/how-to), and search intent. | Yes |
| **build_outline** | Produces the full article blueprint: H1, meta title, meta description, section-by-section word budgets, 3–5 internal link suggestions, and 2–4 external citation placements. | Yes |
| **generate_sections** | Writes each section individually using its heading, description, keyword targets, and word budget from the outline. | Yes |
| **postprocess** | Assembles body markdown, computes keyword density, generates FAQ answers from People Also Ask data using structured output. | Yes (FAQ) |
| **validate_output** | Runs 10 programmatic SEO checks — no LLM. Returns a scored `ValidationResult` attached to the article. | No |

### Design Caveats Worth Noting

**`generated_sections` uses an append reducer.**  
Declared as `Annotated[List[str], operator.add]` in `ArticleState` — LangGraph appends each call's list rather than overwriting. This makes the generate step idempotent if re-run from a checkpoint.

**Word budget is scaled to 93% of target.**  
The LLM consistently overshoots section targets by ~7%, so `build_outline` scales budgets down before writing them into the outline. This keeps the final word count inside the 10% tolerance window that `validate_output` checks.

**`validate_output` is purely programmatic.**  
No LLM call — just regex and arithmetic. This is intentional: validation should be deterministic and fast, not probabilistic. The commented extension point in `builder.py` shows how a revision loop could feed back into `generate_sections` when the score is too low.

> ## 🎙️ SPEAKER NOTES — Pipeline
> - Walk through the mermaid diagram left to right — blue nodes call the LLM, yellow assembles data, green validates.
> - Emphasise that the graph is **linear by design** — no branching today, but there's a commented conditional edge in `builder.py` for a future revision loop.
> - For caveats: the append reducer and word budget scaling are the two non-obvious implementation choices worth calling out — they solve real edge cases found during testing.
> - Say: *"State checkpointing means if the server crashes mid-run, we can call the `/resume` endpoint and LangGraph replays from the last successful node — we don't restart from scratch."*

---
## 3. Live Run

> **Pre-requisite:** The API server must be running before executing these cells.
> ```bash
> cd seo_agent
> uvicorn main:app --reload --port 8000
> ```

Jobs are submitted asynchronously — the API returns a `thread_id` immediately (`202 Accepted`) and the pipeline runs in the background. We poll `/jobs/{thread_id}` to watch the stage progress.

In [ ]:
import requests
import time

BASE_URL = "http://localhost:8000"

In [ ]:
payload = {
    "topic": "best productivity tools for remote teams",
    "target_word_count": 1500,
    "language": "en",
    "use_mock": False   # set True to skip SerpAPI and use built-in mock data
}

resp = requests.post(f"{BASE_URL}/jobs", json=payload)
resp.raise_for_status()
job = resp.json()

thread_id = job["thread_id"]
print(f"✓ Job submitted")
print(f"  thread_id : {thread_id}")
print(f"  status    : {job['status']}")

In [ ]:
STAGES = [
    "serp_fetch",
    "analyze_serp",
    "build_outline",
    "generate_sections",
    "postprocess",
    "validate_output",
]

seen_stages = set()
stage_start = time.time()
poll_count = 0
print("Polling — stages will appear as they complete:\n")

while True:
    resp = requests.get(f"{BASE_URL}/jobs/{thread_id}")
    resp.raise_for_status()
    data = resp.json()
    poll_count += 1

    stage = data.get("execution_stage")
    job_status = data["status"]

    if stage and stage not in seen_stages:
        idx = STAGES.index(stage) + 1 if stage in STAGES else "?"
        elapsed = time.time() - stage_start
        print(f"  [{idx}/{len(STAGES)}] ✓ {stage:<20}  {elapsed:5.1f}s  ({poll_count} poll{'s' if poll_count != 1 else ''})")
        seen_stages.add(stage)
        stage_start = time.time()
        poll_count = 0

    if job_status in ("completed", "failed"):
        print(f"\nJob {job_status.upper()}")
        if data.get("error_message"):
            print(f"Error: {data['error_message']}")
        break

    time.sleep(3)

> ## 🎙️ SPEAKER NOTES — Polling
> - While the poll loop is running, narrate what's happening: *"The job is running in the background — FastAPI's BackgroundTasks — and we're polling every 3 seconds. Each time a node completes, LangGraph fires an `on_chain_end` event which updates the stage in jobs.db."*
> - If generate_sections takes a while: *"This is the longest stage — it's writing each section of the article individually, one LLM call per section, so the model can focus on the right keyword targets and word budget for each heading."*
> - After completion: *"Now let's pull the full result."*

In [ ]:
resp = requests.get(f"{BASE_URL}/jobs/{thread_id}/result")
resp.raise_for_status()
result = resp.json()

print(f"Result fetched — {len(result)} top-level keys")
print(list(result.keys()))

---
## 4. Results Walkthrough

The response is a single `ArticleOutput` object with three logical groups:

| Group | Keys |
|---|---|
| **Article Content** | `h1`, `meta_title`, `meta_description`, `body_markdown`, `word_count` |
| **SEO Intelligence** | `keyword_analysis`, `internal_links`, `external_references`, `faq` |
| **Quality Gate** | `validation_results` |

### 4.1 Article Content

In [ ]:
print("H1")
print(f"  {result['h1']}")
print()
print("Meta Title")
print(f"  {result['meta_title']}  ({len(result['meta_title'])} chars)")
print()
print("Meta Description")
print(f"  {result['meta_description']}  ({len(result['meta_description'])} chars)")
print()
print(f"Word Count:  {result['word_count']}  (target: 1500)")

In [ ]:
# Show the opening of the article body
preview = result["body_markdown"][:800]
print(preview)
print("\n[... article continues ...]")

### 4.2 SEO Intelligence

In [ ]:
ka = result["keyword_analysis"]

print("── Keyword Analysis ─────────────────────────────────────────────────")
print(f"  Primary keyword : '{ka['primary_keyword']}'")
print(f"  Occurrences     : {ka['primary_keyword_count']}")
print(f"  Density         : {ka['keyword_density']:.2%}")
print(f"  Secondary KWs   : {len(ka['secondary_keywords'])} tracked")
for kw in ka["secondary_keywords"][:4]:
    sections = ", ".join(kw["sections_present"][:2]) or "—"
    print(f"    · '{kw['keyword']}' — {kw['count']} occurrences  |  in: {sections}")
print()

print("── Internal Links ───────────────────────────────────────────────────")
for link in result["internal_links"]:
    print(f"  · \"{link['anchor_text']}\"")
    print(f"    → target: {link['suggested_target_topic']}")
print()

print("── External References ──────────────────────────────────────────────")
for ref in result["external_references"]:
    print(f"  · {ref['source_name']}")
    print(f"    section: {ref['placement_section']}")
print()

print("── FAQ ──────────────────────────────────────────────────────────────")
for item in result["faq"]:
    print(f"  Q: {item['question']}")
    print(f"  A: {item['answer'][:120]}...")
    print()

### 4.3 Validation Layer

10 programmatic SEO checks — no LLM involved. Each check is deterministic: regex, character counts, arithmetic. The overall score is the percentage of checks passed.

In [ ]:
vr = result["validation_results"]
score = vr["overall_score"]
overall_icon = "✅" if vr["passed"] else "⚠️"

print(f"Overall Score: {score}/100  {overall_icon}")
print()
print(f"{'Check':<38}  {'Result':<6}  Detail")
print("─" * 90)
for check in vr["checks"]:
    icon = "✅" if check["passed"] else "❌"
    print(f"{check['check_name']:<38}  {icon:<6}  {check['detail']}")

> ## 🎙️ SPEAKER NOTES — Results
> - For Article Content: *"The pipeline produces structured metadata alongside the body — the meta title and description are character-constrained by the postprocess node before they land in the output."*
> - For SEO Intelligence: *"This is where the SERP analysis pays off — the agent identified what keyword to rank for, tracked its density, suggested contextually appropriate internal links, and answered real People Also Ask questions for the FAQ."*
> - For Validation: *"The validation node is entirely programmatic — 10 checks, no LLM. Any failures flag exactly what to fix. The extension point in the graph is a conditional edge that would re-enter generate_sections when the score is too low."*
> - If there are failures: *"Any failures here are clear, actionable signals — the check names and detail fields tell you exactly what the constraint is and by how much it missed."*
> - Close with: *"The full output is a single JSON object — structured, schema-validated by Pydantic, and ready to hand off to a CMS or post-processing step."*